# Precision Intervention & Resource Allocation: How Do We Fix It?
## CARB-X x Gates Foundation Strategic Initiative | Notebook 09

**Notebook 08 identified who is being left behind. Notebook 09 answers the harder question: what specifically do we deploy, where, and why?**

Every global health proposal eventually reaches the same wall: the intervention is right,
the evidence is compelling, the need is undeniable — and then a programme officer asks:
*"But is this the right tool for this specific patient in this specific context?"*

This notebook builds the **Precision Intervention Matrix** — a country-level prescription
that synthesises all findings from Notebooks 01-08 into a single, actionable brief.

Seven questions are addressed, each grounded in a specific finding from the prior analysis:

1. **Reactivation vs Re-infection** — Is the burden driven by malnutrition-triggered latency, or constant household re-transmission? This determines whether we deploy diagnostics or fund nutrition.
2. **AI Blind Spots** — In which countries does the Notebook 06 CV model carry a systematic false-negative risk due to atypical HIV/TB presentations?
3. **The Stigma Channel** — Where does the male healthcare-seeking gap map onto private pharmacy dominance?
4. **UNHLM Accountability** — Which governments have made the structural commitments (social protection, food security, free diagnostics) that make them viable long-term partners?

**Two datasets enter the pipeline for the first time:**
- `LTBI_estimates` — modelled household contact counts and paediatric TPT eligibility
- `TB_unhlm` — UN High-Level Meeting political commitments and social protection frameworks

**Author's Note:**
This notebook is deliberately prescriptive. The findings of Notebooks 01-08 are not
arguments — they are instructions. The goal is to give a programme officer a single
page they can take into a funding meeting.


In [ ]:
# Imports & Visual Theme
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DARK_BG    = '#0F1117'
TEXT_WHITE = '#FFFFFF'
TEXT_GREY  = '#AAAAAA'
TEXT_DIM   = '#666677'
TEAL       = '#2A9D8F'
AMBER      = '#F4A261'
RED        = '#E63946'
BLUE       = '#457B9D'
PURPLE     = '#6A4C93'
YELLOW     = '#FFD166'
PINK       = '#E76F9A'
GREEN      = '#06D6A0'

plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'axes.facecolor':     DARK_BG,
    'figure.facecolor':   DARK_BG,
    'text.color':         TEXT_WHITE,
    'axes.labelcolor':    '#CCCCCC',
    'xtick.color':        '#CCCCCC',
    'ytick.color':        '#CCCCCC',
    'axes.grid':          True,
    'grid.color':         '#1E1E2E',
    'grid.linewidth':     0.7,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.spines.left':   False,
    'axes.spines.bottom': False,
})

BASE_DIR = Path('/Users/fridaarrey/Desktop/WHO_TB_Project')
RAW      = BASE_DIR / 'WHO_raw_csv'
FIG_PATH = BASE_DIR / 'figures'
FIG_PATH.mkdir(exist_ok=True)

HBC = [
    'IND','CHN','IDN','PHL','PAK','NGA','ZAF','BGD','COD','MMR',
    'ETH','TZA','MOZ','AGO','PRK','VNM','UGA','RUS','KEN','ZWE',
    'ZMB','BRA','LSO','NAM','GAB','GNQ','PNG','SLE','THA','CAF'
]
REGION_COLOURS = {
    'AFR': RED, 'SEA': AMBER, 'EMR': TEAL,
    'WPR': BLUE, 'AMR': PURPLE, 'EUR': '#8D99AE'
}

print("Notebook 09 initialised.")
print("Author: Dr. Frida Arrey Takubetang")
print()
print("Two datasets entering the pipeline for the first time:")
print("  LTBI_estimates   (household contacts, paediatric TPT eligibility)")
print("  TB_unhlm         (UNHLM political commitments, social protection)")
print()
print("All 16 WHO datasets have now been used across this pipeline.")


In [ ]:
# Load All Data
df_burden = pd.read_csv(RAW / 'WHO TB dataset_2024-03-21.xlsx - TB_burden_countries_2024-03-21.csv')
df_age    = pd.read_csv(RAW / 'WHO TB dataset_2024-03-21.xlsx - TB_burden_age_sex_2024-03-21.csv')
df_out_as = pd.read_csv(RAW / 'WHO TB dataset_2024-03-21.xlsx - TB_outcomes_age_sex_2024-03-21.csv')
df_tpt    = pd.read_csv(RAW / 'WHO TB dataset_2024-03-21.xlsx - TB_contact_tpt_2024-03-21.csv')
df_hiv_nr = pd.read_csv(RAW / 'WHO TB dataset_2024-03-21.xlsx - TB_hiv_nonroutine_surveillance_.csv')
df_ppm    = pd.read_csv(RAW / 'WHO TB dataset_2024-03-21.xlsx - TB_ppm_2024-03-21.csv')
df_budget = pd.read_csv(RAW / 'WHO TB dataset_2024-03-21.xlsx - TB_budget_2024-03-21.csv')

# Two new files -- first use in the entire pipeline
df_ltbi  = pd.read_csv(RAW / 'WHO TB dataset_2024-03-21.xlsx - LTBI_estimates_2024-03-21.csv')
df_unhlm = pd.read_csv(RAW / 'WHO TB dataset_2024-03-21.xlsx - TB_unhlm_2024-03-21.csv')

region_map  = df_burden[['iso3', 'g_whoregion']].drop_duplicates()
country_map = df_burden[['iso3', 'country']].drop_duplicates()

print(f"LTBI_estimates: {df_ltbi.shape}  -- household contact modelling, 2022")
print(f"  Key fields: e_hh_contacts, e_prevtx_eligible, e_prevtx_kids_pct")
print()
print(f"TB_unhlm:       {df_unhlm.shape}  -- UNHLM reporting, 2023")
print(f"  Key fields: social_protn, food_security, free_access_tbdx, min_agg_collab")
print(f"  Codebook: min_agg_collab 231 = TB prevention + care, 232 = patient support incl. nutrition")


In [ ]:
# Build Precision Master Table

# Undernutrition driver
und_d = df_age[(df_age['age_group']=='all') & (df_age['sex']=='a')].pivot_table(
    index='iso3', columns='risk_factor', values='best', aggfunc='sum')
und_d['und_pct'] = (und_d['und'] / und_d['all'] * 100).fillna(0)

# Gender gap
gg = (df_out_as[df_out_as['age_group']=='a']
      .groupby(['iso3','sex'])['tsr'].mean().unstack().dropna())
gg['gender_gap'] = (gg['f'] - gg['m']).round(1)
gg = gg.reset_index().rename(columns={'f':'tsr_f','m':'tsr_m'})

# TPT efficiency
tpt22 = df_tpt[df_tpt['year']==2022].copy()
tpt22['tpt_eff'] = (tpt22['newinc_con_prevtx'] /
                    tpt22['newinc_con_screen'].replace(0, np.nan) * 100)

# HIV undercount (e_tbhiv_prct is already in percent -- no x100 conversion)
df_b22 = df_burden[df_burden['year']==2022][['iso3','country','g_whoregion','e_tbhiv_prct']].copy()
df_b22.rename(columns={'e_tbhiv_prct':'routine_pct'}, inplace=True)
hiv_lat = (df_hiv_nr.sort_values('year').groupby('iso3').last().reset_index()
           [['iso3','tbhiv_surv_prev','tbhiv_sentin_prev','year']]
           .rename(columns={'year':'survey_year'}))
hiv_lat['survey_pct'] = hiv_lat['tbhiv_surv_prev'].fillna(hiv_lat['tbhiv_sentin_prev'])
merged = df_b22.merge(hiv_lat, on='iso3', how='inner').dropna(subset=['survey_pct','routine_pct'])
merged['undercount_gap'] = (merged['survey_pct'] - merged['routine_pct']).clip(lower=0)

# Private sector mix
ppm22 = df_ppm[df_ppm['year']==2022][['iso3','priv_new_dx','pub_new_dx']].dropna()
ppm22['total_dx'] = ppm22['priv_new_dx'] + ppm22['pub_new_dx']
ppm22['priv_pct'] = (ppm22['priv_new_dx'] / ppm22['total_dx'] * 100).fillna(0)

# Budget domestic funding
b22 = df_budget[df_budget['year']==2022][
    ['iso3','budget_tot','cf_tot','cf_tot_domestic','cf_tot_gf']].copy()
b22['domestic_pct']    = (b22['cf_tot_domestic'] / b22['cf_tot'] * 100).round(1)
b22['funding_gap_pct'] = ((b22['budget_tot']-b22['cf_tot']) / b22['budget_tot'] * 100).clip(lower=0).round(1)

# UNHLM social protection pillars
# min_agg_collab: 231 = TB prevention and care, 232 = patient support (inc. nutrition)
#                 230 = advocacy only, 6 = not engaged
unhlm_sp = df_unhlm[['iso3','social_protn','food_security','free_access_tbdx',
                      'ms_review','min_agg_collab']].copy()
unhlm_sp['agri_engaged']      = unhlm_sp['min_agg_collab'].apply(lambda x: 1 if x in [231,232] else 0)
unhlm_sp['has_social_protn']  = (unhlm_sp['social_protn']==1).astype(int)
unhlm_sp['has_food_security'] = (unhlm_sp['food_security']==1).astype(int)
unhlm_sp['has_free_dx']       = (unhlm_sp['free_access_tbdx']==1).astype(int)

# LTBI paediatric estimates
ltbi22 = df_ltbi[df_ltbi['year']==2022][
    ['iso3','e_prevtx_eligible','newinc_con_prevtx','e_prevtx_kids_pct']].copy()
ltbi22['paed_pct_elig'] = ltbi22['e_prevtx_kids_pct'].fillna(0)
ltbi22['tpt_coverage']  = (ltbi22['newinc_con_prevtx'] /
                           ltbi22['e_prevtx_eligible'].replace(0, np.nan) * 100).fillna(0)

# Assemble master table
eq = country_map[country_map['iso3'].isin(HBC)].copy()
eq = (eq
    .merge(und_d[['und_pct']].reset_index(),   on='iso3', how='left')
    .merge(gg[['iso3','gender_gap','tsr_f','tsr_m']], on='iso3', how='left')
    .merge(tpt22[['iso3','tpt_eff']],           on='iso3', how='left')
    .merge(merged[['iso3','routine_pct','undercount_gap']], on='iso3', how='left')
    .merge(ppm22[['iso3','priv_pct']],          on='iso3', how='left')
    .merge(b22[['iso3','domestic_pct','funding_gap_pct']], on='iso3', how='left')
    .merge(unhlm_sp,                            on='iso3', how='left')
    .merge(ltbi22[['iso3','tpt_coverage','paed_pct_elig']], on='iso3', how='left')
    .merge(region_map,                          on='iso3', how='left')
)

# Equity score (Notebook 08 methodology)
for col in ['und_pct','gender_gap','undercount_gap']:
    cm = eq[col].max()
    eq[f'{col}_n'] = (eq[col]/cm*100).fillna(0) if cm and cm>0 else 0.0
tpt_min, tpt_max = eq['tpt_eff'].min(), eq['tpt_eff'].max()
eq['tpt_vuln_n'] = ((tpt_max-eq['tpt_eff'])/(tpt_max-tpt_min)*100).fillna(50)
eq['equity_score'] = (
    eq['und_pct_n'].fillna(0)*0.35 +
    eq['gender_gap_n'].fillna(0)*0.25 +
    eq['tpt_vuln_n'].fillna(0)*0.25 +
    eq['undercount_gap_n'].fillna(0)*0.15
)

# Political will score
eq['political_will'] = (
    eq['has_social_protn'].fillna(0) +
    eq['has_food_security'].fillna(0) +
    eq['has_free_dx'].fillna(0) +
    eq['agri_engaged'].fillna(0)
)

def prescribe(row):
    p = []
    if row.get('und_pct', 0) > 35:                                      p.append('NUTRITION-DX BUNDLE')
    if row.get('undercount_gap', 0) > 15:                               p.append('DUAL HIV+TB MOLECULAR DX')
    if row.get('gender_gap', 0) > 3 and row.get('priv_pct', 0) > 50:   p.append('PHARMACY AI TRIAGE')
    if row.get('tpt_eff', 100) < 30:                                    p.append('DIGITAL TPT TRACKER')
    if row.get('domestic_pct', 100) < 30:                               p.append('EMERGENCY GRANT')
    if row.get('has_free_dx', 1) == 0:                                  p.append('FREE-DX POLICY ADVOCACY')
    return ' | '.join(p) if p else 'STANDARD DEPLOYMENT'

eq['prescription'] = eq.apply(prescribe, axis=1)
eq = eq.sort_values('equity_score', ascending=False)

print(f"Precision master table: {eq.shape[0]} countries x {eq.shape[1]} features")
print()
print("=" * 80)
print("PRECISION BRIEF -- TOP 12 PRIORITY COUNTRIES")
print("=" * 80)
for _, row in eq.head(12).iterrows():
    print(f"  {row['country']:<30} Score:{row['equity_score']:>5.1f}  {row['prescription']}")


---
## Section 1 -- Questions 1 & 5: The Biological Driver
### Reactivation vs Re-infection

**The clinical distinction (Dr. Arrey):**
When a person develops active TB, one of two things happened:

**(A) Endogenous Reactivation** -- they were infected years ago, their immune system
contained the bacteria in latent form, and then chronic malnutrition degraded
T-cell function sufficiently to allow reactivation.

**(B) Exogenous Re-infection** -- they were recently exposed to a new infectious source,
typically a household contact with untreated pulmonary TB.

These require fundamentally different interventions:
- Reactivation dominant -> **LTBI screening + nutritional support**
- Transmission dominant -> **case-finding, ventilation, and household management**

**The threshold rule:** Countries where undernutrition drives >35% of TB burden are
biologically in the reactivation zone. In CAF (53%), DRC (47%), North Korea (48%),
Lesotho (44%), and Mozambique (42%), a standalone diagnostic tool will have
structurally limited impact -- not because the AI is inadequate, but because the
patient cannot mount an immune response regardless.

Deploying the most advanced AI diagnostics without addressing the immunological substrate
is like diagnosing a fire while standing next to the fuel tank.


In [ ]:
# Section 1: Reactivation vs Transmission Analysis

react = (und_d[['und_pct']].reset_index()
         .merge(ltbi22, on='iso3', how='inner')
         .merge(country_map, on='iso3', how='left'))
react['driver'] = react['und_pct'].apply(
    lambda x: 'REACTIVATION' if x>35 else ('MIXED' if x>20 else 'TRANSMISSION'))

print("=" * 72)
print("QUESTIONS 1 & 5: BIOLOGICAL DRIVER -- REACTIVATION vs TRANSMISSION")
print("=" * 72)
print(f"{'Country':<28} {'Und%':>6}  {'TPT Cover%':>10}  {'Driver':<14}  {'Strategic Need'}")
print("-" * 70)
for _, row in react[react['iso3'].isin(HBC)].sort_values('und_pct', ascending=False).iterrows():
    need = ('Nutrition + Latent Screening' if row['driver']=='REACTIVATION' else
            'Mixed Bundle' if row['driver']=='MIXED' else 'Case-finding + Ventilation')
    tpt  = f"{row['tpt_coverage']:.0f}%" if pd.notna(row['tpt_coverage']) else 'n/a'
    flag = '>> ' if row['driver']=='REACTIVATION' else '   '
    print(f"{flag}{row['country']:<25} {row['und_pct']:>5.1f}%  {tpt:>10}  {row['driver']:<14}  {need}")
print("=" * 72)


In [ ]:
# Chart 1: Reactivation vs Transmission

react_hbc = react[react['iso3'].isin(HBC)].dropna(subset=['und_pct','tpt_coverage'])
fig, ax = plt.subplots(figsize=(13, 8))
fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(DARK_BG)

colours = react_hbc['und_pct'].apply(lambda x: RED if x>35 else (AMBER if x>20 else TEAL))
ax.scatter(react_hbc['und_pct'], react_hbc['tpt_coverage'].clip(upper=200),
           c=colours, s=180, alpha=0.85, edgecolors='white', linewidths=0.7, zorder=4)
for _, row in react_hbc.iterrows():
    ax.annotate(row['country'], xy=(row['und_pct'], min(row['tpt_coverage'],200)),
                xytext=(row['und_pct']+0.5, min(row['tpt_coverage'],200)+3),
                fontsize=7.5, color='white', alpha=0.85)

ax.axvline(35, color=RED, lw=1.2, linestyle='--', alpha=0.5)
ax.axvline(20, color=AMBER, lw=1, linestyle=':', alpha=0.4)
ax.axhline(40, color=YELLOW, lw=1, linestyle=':', alpha=0.4)
ax.text(36.5, 5, 'REACTIVATION ZONE', color=RED, fontsize=9, alpha=0.8, fontweight='bold')
ax.text(0.5,  5, 'TRANSMISSION ZONE', color=TEAL, fontsize=9, alpha=0.8, fontweight='bold')
ax.text(0.5, 185, '< Good TPT coverage (>40%)', color=YELLOW, fontsize=8.5, alpha=0.7)
ax.text(0.5,  30, '< DANGER: Low coverage + High reactivation risk', color=RED, fontsize=8.5, alpha=0.7)

legend_elems = [
    mpatches.Patch(color=RED,   label='Reactivation dominant (>35% undernutrition)', alpha=0.85),
    mpatches.Patch(color=AMBER, label='Mixed driver (20-35%)', alpha=0.85),
    mpatches.Patch(color=TEAL,  label='Transmission dominant (<20%)', alpha=0.85),
]
ax.legend(handles=legend_elems, fontsize=9, framealpha=0.15, edgecolor='#444', loc='upper right')
ax.set_xlabel('Undernutrition-Attributable TB (%)', fontsize=10)
ax.set_ylabel('TPT Coverage of Eligible Contacts (%)', fontsize=10)
ax.tick_params(length=0)

fig.text(0.5, 0.97, 'Q1 + Q5: Biological Driver -- Reactivation vs Transmission',
         ha='center', fontsize=13, fontweight='bold', color=TEXT_WHITE)
fig.text(0.5, 0.93,
         'Countries in the danger zone need Nutrition+Diagnostics bundles -- deploying AI X-ray tools alone will not fix a malnutrition-driven epidemic',
         ha='center', fontsize=10, color=TEXT_GREY)
fig.text(0.5, 0.01, 'Sources: TB_burden_age_sex + LTBI_estimates_2024-03-21 (LTBI file first use)',
         ha='center', fontsize=8, color=TEXT_DIM)
plt.tight_layout(rect=[0, 0.04, 1, 0.92])
plt.savefig(FIG_PATH / '09_reactivation_vs_transmission.png', dpi=180, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print("Saved: figures/09_reactivation_vs_transmission.png")


---
## Section 2 -- Questions 2 & 6: The AI Blind Spot
### Where the Notebook 06 Computer Vision Model Has Systematic False-Negative Bias

**The immunological mechanism (Dr. Arrey):**
HIV-positive patients with TB frequently present with **paucibacillary disease** --
fewer bacteria, less cavitation, less consolidation, more diffuse infiltrates or
entirely normal chest X-rays. The immune system of an HIV-positive patient cannot
mount the granulomatous response that produces the classic radiological TB signature.

The AI correctly interprets what it sees. The problem is that what it sees is not
the full picture.

The Diagnostic Reliability Index (DRI) translates the HIV undercount gap directly
into AI deployment risk:
- Undercount gap >15pp: **HIGH** -- mandatory GeneXpert follow-up required
- Undercount gap 5-15pp: **MODERATE** -- AI + rapid HIV test
- Undercount gap <5pp: **LOW** -- standard AI triage acceptable

**The honest admission:** In Mozambique (+31pp), Tanzania (+30pp), Namibia (+25pp),
Uganda (+21pp), and Kenya (+17pp), the Notebook 06 AI model should not be deployed
as a standalone diagnostic. Saying this demonstrates scientific integrity.


In [ ]:
# Section 2: AI Diagnostic Reliability Index

ai_audit = eq[['iso3','country','routine_pct','undercount_gap','g_whoregion']].dropna(subset=['undercount_gap']).copy()
ai_audit['ai_risk'] = ai_audit['undercount_gap'].apply(
    lambda x: 'HIGH'     if x > 15 else
              'MODERATE' if x >  5 else 'LOW')
ai_audit['protocol'] = ai_audit['ai_risk'].map({
    'HIGH':     'AI triage + mandatory GeneXpert follow-up',
    'MODERATE': 'AI triage + HIV rapid test at point of care',
    'LOW':      'Standard AI triage acceptable',
})

print("=" * 78)
print("QUESTION 2 + 6: AI DIAGNOSTIC BLIND SPOT AUDIT")
print("=" * 78)
for _, row in ai_audit.sort_values('undercount_gap', ascending=False).head(15).iterrows():
    flag = '[HIGH]    ' if row['ai_risk']=='HIGH' else ('[MODERATE] ' if row['ai_risk']=='MODERATE' else '[LOW]      ')
    uc   = f"+{row['undercount_gap']:.1f}pp" if row['undercount_gap']>0 else f"{row['undercount_gap']:.1f}pp"
    print(f"  {flag} {row['country']:<28} {row['routine_pct']:>6.1f}% routine  {uc:>8} gap  |  {row['protocol']}")
print()
print(f"Summary: {(ai_audit['ai_risk']=='HIGH').sum()} HIGH  /  "
      f"{(ai_audit['ai_risk']=='MODERATE').sum()} MODERATE  /  "
      f"{(ai_audit['ai_risk']=='LOW').sum()} LOW risk countries")


In [ ]:
# Chart 2: AI Blind Spot Risk

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor(DARK_BG)

ax = axes[0]; ax.set_facecolor(DARK_BG)
risk_col = ai_audit['undercount_gap'].apply(lambda x: RED if x>15 else (AMBER if x>5 else TEAL))
ax.scatter(ai_audit['routine_pct'], ai_audit['undercount_gap'],
           c=risk_col, s=160, alpha=0.88, edgecolors='white', linewidths=0.7, zorder=4)
for _, row in ai_audit.iterrows():
    ax.annotate(row['country'], xy=(row['routine_pct'],row['undercount_gap']),
                xytext=(row['routine_pct']+0.3, row['undercount_gap']+0.5),
                fontsize=7.5, color='white', alpha=0.85)
ax.axhline(15, color=RED,   lw=1.2, linestyle='--', alpha=0.5)
ax.axhline(5,  color=AMBER, lw=1.0, linestyle=':',  alpha=0.4)
ax.text(0.2, 16.5, 'HIGH BLIND-SPOT RISK -- Mandatory GeneXpert follow-up',
        color=RED, fontsize=8.5, fontweight='bold', alpha=0.85)
ax.text(0.2, 6.5,  'MODERATE -- AI triage + HIV rapid test',
        color=AMBER, fontsize=8.5, alpha=0.85)
ax.set_xlabel('Routine HIV/TB Estimate (%)', fontsize=10)
ax.set_ylabel('Surveillance Undercount Gap (pp)', fontsize=10)
ax.set_title('AI Blind Spot Risk:\nActual HIV/TB vs Routine Data', color=TEXT_WHITE, fontsize=11, pad=8)
ax.tick_params(length=0)

ax2 = axes[1]; ax2.set_facecolor(DARK_BG)
risk_cats = {
    'HIGH\n(>15pp gap)':  (ai_audit['undercount_gap']>15).sum(),
    'MODERATE\n(5-15pp)': ((ai_audit['undercount_gap']>5)&(ai_audit['undercount_gap']<=15)).sum(),
    'LOW\n(<5pp)':        (ai_audit['undercount_gap']<=5).sum(),
}
bars = ax2.bar(risk_cats.keys(), risk_cats.values(),
               color=[RED, AMBER, TEAL], alpha=0.85, width=0.5, zorder=3,
               edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, risk_cats.values()):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, str(int(val)),
             ha='center', fontsize=16, color='white', fontweight='bold')
ax2.yaxis.set_visible(False); ax2.tick_params(length=0)
ax2.set_title('HBC Countries by\nAI Diagnostic Reliability Risk', color=TEXT_WHITE, fontsize=11, pad=8)
ax2.text(1, 3.8,
         'In HIGH-risk countries, HIV+ TB\npatients present with atypical X-rays.\nThe AI model has a systematic\nfalse-negative bias here.',
         ha='center', va='bottom', fontsize=9, color=TEXT_GREY,
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#1E1E2E', edgecolor='#444', alpha=0.9))

fig.text(0.5, 0.97, 'Q2 + Q6: AI Diagnostic Blind Spots -- Where X-Ray Alone Fails',
         ha='center', fontsize=13, fontweight='bold', color=TEXT_WHITE)
fig.text(0.5, 0.93,
         'This is not a flaw in the model -- it is a biological limit. Identifying it is what separates a deployment proposal from a marketing document.',
         ha='center', fontsize=10, color=TEXT_GREY)
fig.text(0.5, 0.01, 'Sources: TB_hiv_nonroutine_surveillance + TB_burden_countries_2024',
         ha='center', fontsize=8, color=TEXT_DIM)
plt.tight_layout(rect=[0, 0.04, 1, 0.92])
plt.savefig(FIG_PATH / '09_ai_blind_spot_risk.png', dpi=180, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print("Saved: figures/09_ai_blind_spot_risk.png")


---
## Section 3 -- Question 3: The Stigma Channel
### Private Sector Dominance as a Male Healthcare Access Problem

**The access mechanism (Dr. Arrey):**
In Nigeria, Pakistan, and India, the private sector accounts for 93-98% of TB diagnoses.
Men are not refusing healthcare. They are seeking it informally. The TB stigma is severe:
a public clinic TB diagnosis means community members know within days. A private pharmacy
consultation does not.

When private sector dominance is high AND the gender treatment gap is positive, the
intervention is not awareness campaigns or more clinics. It is a diagnostic that goes
to where men already seek care: **Pharmacy-linked AI triage**.

A mobile-based triage tool that allows a private pharmacist to screen for TB, generate
a referral, and notify a community health worker without requiring the patient to
visit a public clinic.


In [ ]:
# Section 3: PPM & Gender Gap Analysis

ppm_gender = eq[['iso3','country','priv_pct','gender_gap','g_whoregion']].dropna(subset=['priv_pct','gender_gap'])
corr_val = ppm_gender[['priv_pct','gender_gap']].corr().iloc[0,1]
print(f"Pearson correlation (priv_pct vs gender_gap): {corr_val:.3f}")
print()
print("Countries where private sector dominates AND gender gap exists:")
print(f"{'Country':<26} {'Priv%':>7}  {'Gender Gap':>10}  {'Channel'}")
print("-" * 72)
for _, row in ppm_gender.sort_values('priv_pct',ascending=False).iterrows():
    if row['priv_pct'] > 50 and row['gender_gap'] > 2:
        channel = 'PHARMACY AI TRIAGE -- men seeking informal care'
    elif row['priv_pct'] > 50:
        channel = 'Pharmacy integration (gap manageable)'
    else:
        channel = 'Standard clinic outreach'
    flag = '>> ' if row['priv_pct']>50 and row['gender_gap']>2 else '   '
    print(f"{flag}{row['country']:<23} {row['priv_pct']:>6.0f}%  {row['gender_gap']:>+9.1f}pp  {channel}")


In [ ]:
# Chart 3: PPM & Gender -- The Pharmacy Channel

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor(DARK_BG)

ax = axes[0]; ax.set_facecolor(DARK_BG)
for region, grp in ppm_gender.groupby('g_whoregion'):
    col = REGION_COLOURS.get(region,'#888')
    ax.scatter(grp['priv_pct'], grp['gender_gap'], c=col, s=160, alpha=0.85,
               edgecolors='white', linewidths=0.7, zorder=4)
for _, row in ppm_gender.iterrows():
    ax.annotate(row['country'], xy=(row['priv_pct'],row['gender_gap']),
                xytext=(row['priv_pct']+1, row['gender_gap']+0.15),
                fontsize=7.5, color='white', alpha=0.85)
ax.axvline(50, color=YELLOW, lw=1, linestyle=':', alpha=0.5)
ax.axhline(3,  color=PINK,   lw=1, linestyle='--', alpha=0.5)
ax.fill_between([50,105],[3,3],[12,12], alpha=0.06, color=PINK)
ax.text(52, 3.5, 'PHARMACY AI TRIAGE\nHigh private + High gap',
        color=PINK, fontsize=8, alpha=0.9)
ax.set_xlabel('Private Sector TB Diagnoses (%)', fontsize=10)
ax.set_ylabel('Gender Treatment Gap (pp, F minus M)', fontsize=10)
ax.set_title('Private Sector vs Gender Gap:\nThe Pharmacy Triage Opportunity', color=TEXT_WHITE, fontsize=11, pad=8)
ax.tick_params(length=0)

ax2 = axes[1]; ax2.set_facecolor(DARK_BG)
top8 = ppm_gender.sort_values('priv_pct', ascending=False).head(8)
y_pos = np.arange(len(top8))
ax2.barh(y_pos, top8['priv_pct'], color=BLUE, alpha=0.8, height=0.6, zorder=3)
ax2.axvline(50, color=YELLOW, lw=1, linestyle=':', alpha=0.5)
for i, (_, row) in enumerate(top8.iterrows()):
    col = PINK if row['gender_gap']>3 else (AMBER if row['gender_gap']>1 else TEAL)
    ax2.text(row['priv_pct']+1, i, f"Gender gap: {row['gender_gap']:+.1f}pp",
             va='center', fontsize=8, color=col, alpha=0.9)
ax2.set_yticks(y_pos); ax2.set_yticklabels(top8['country'], fontsize=9)
ax2.set_xlabel('Private Sector Share of TB Diagnoses (%)', fontsize=10)
ax2.set_title('Countries Where Men Seek\nInformal TB Care First', color=TEXT_WHITE, fontsize=11, pad=8)
ax2.tick_params(length=0); ax2.set_xlim(0, 115)

fig.text(0.5, 0.97, 'Q3 + Q7: The Stigma Channel -- Why Men Choose Pharmacies Over TB Clinics',
         ha='center', fontsize=13, fontweight='bold', color=TEXT_WHITE)
fig.text(0.5, 0.93,
         'Where the private sector dominates TB diagnosis, pharmacy-linked AI triage is the structural fix for the gender gap',
         ha='center', fontsize=10, color=TEXT_GREY)
fig.text(0.5, 0.01, 'Sources: TB_ppm_2024 + TB_outcomes_age_sex_2024',
         ha='center', fontsize=8, color=TEXT_DIM)
plt.tight_layout(rect=[0, 0.04, 1, 0.92])
plt.savefig(FIG_PATH / '09_ppm_gender_pharmacy.png', dpi=180, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print("Saved: figures/09_ppm_gender_pharmacy.png")


---
## Section 4 -- UNHLM Accountability: Funding Gaps & Political Will
### Where Gates Foundation Is Substituting for Absent Domestic Investment

**The sustainability problem (Dr. Arrey):**
The UN High-Level Meeting on TB (2018, 2023) produced binding commitments from heads
of state: social protection for TB patients, free access to diagnostics, ministry of
agriculture linkage for nutrition interventions.

The budget data tells a different story. DRC (6.3% domestic), Zimbabwe (6.7%),
Pakistan (6.9%), Myanmar (9.7%) are allowing the Global Fund and USAID to carry 90%+
of the operational cost.

This matters for two reasons:

**Deployment sustainability:** A diagnostic tool in a country with 6% domestic
co-funding will be abandoned when the grant ends.

**Partner selection:** The UNHLM social protection data identifies which governments
have made structural commitments that make the investment sustainable beyond the
grant period.

**The prescription:** Where domestic funding is below 30% AND UNHLM commitments
are absent, the recommendation shifts from "co-fund" to "emergency grant with
conditionality" -- funding tied to domestic commitment milestones.


In [ ]:
# Section 4: UNHLM Accountability & Funding Gap

fund_audit = eq[['iso3','country','domestic_pct','funding_gap_pct',
                 'has_social_protn','has_food_security','has_free_dx','agri_engaged',
                 'political_will','equity_score']].dropna(subset=['domestic_pct']).copy()

def funding_strategy(row):
    if row['domestic_pct'] < 30:
        return 'EMERGENCY GRANT with conditionality'
    elif row['domestic_pct'] < 60 and row['political_will'] < 2:
        return 'Co-fund + political alignment required'
    elif row['political_will'] >= 3:
        return 'Partnership -- govt is viable co-investor'
    return 'Standard co-funding'

fund_audit['funding_strategy'] = fund_audit.apply(funding_strategy, axis=1)

print("=" * 80)
print("UNHLM ACCOUNTABILITY -- FUNDING GAP & POLITICAL WILL")
print("=" * 80)
print(f"{'Country':<26} {'Domestic%':>9}  {'PW':>3}/4  {'Gates Strategy'}")
print("-" * 78)
for _, row in fund_audit.sort_values('domestic_pct').iterrows():
    flag = '[!] ' if row['domestic_pct'] < 30 else '    '
    print(f"{flag}{row['country']:<22} {row['domestic_pct']:>8.0f}%  {int(row['political_will']):>3}/4  {row['funding_strategy']}")
print()
print("Political Will Score = social protection + food security + free DX + agri ministry")
print("Source: TB_unhlm_2024 (first use in this project)")


In [ ]:
# Chart 4: UNHLM Accountability

fund_plot = fund_audit.sort_values('domestic_pct')
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
fig.patch.set_facecolor(DARK_BG)

ax = axes[0]; ax.set_facecolor(DARK_BG)
y_pos = np.arange(len(fund_plot))
bar_col = fund_plot['domestic_pct'].apply(lambda x: RED if x<30 else (AMBER if x<60 else GREEN))
ax.barh(y_pos, fund_plot['domestic_pct'], color=bar_col, alpha=0.85, height=0.65, zorder=3)
ax.axvline(30, color=RED, lw=1.5, linestyle='--', alpha=0.7)
ax.axvline(60, color=AMBER, lw=1, linestyle=':', alpha=0.5)
ax.text(30.5, len(fund_plot)-0.5, '30% threshold', color=RED, fontsize=8, alpha=0.8)
for i, (_, row) in enumerate(fund_plot.iterrows()):
    ax.text(row['domestic_pct']+0.5, i, f"{row['domestic_pct']:.0f}%",
            va='center', fontsize=7.5, color='white', alpha=0.85)
ax.set_yticks(y_pos); ax.set_yticklabels(fund_plot['country'], fontsize=8.5)
ax.set_xlabel('Domestic Funding as % of Available TB Budget', fontsize=10)
ax.set_title('Fiscal Accountability:\nDomestic Co-investment by Country (2022)', color=TEXT_WHITE, fontsize=11, pad=8)
ax.tick_params(length=0); ax.set_xlim(0, 115)
ax.fill_between([0,30],[-.5,-.5],[len(fund_plot)-.5]*2, alpha=0.05, color=RED)
ax.text(1, 1, 'EMERGENCY\nGRANT ZONE', color=RED, fontsize=7.5, alpha=0.7, fontweight='bold')

ax2 = axes[1]; ax2.set_facecolor(DARK_BG)
pillar_cols = ['has_social_protn','has_food_security','has_free_dx','agri_engaged']
pillars     = ['Social\nProtection','Food\nSecurity','Free TB\nDiagnostics','Agriculture\nMinistry Link']
n_total     = len(eq)
scores      = [eq[c].fillna(0).sum() for c in pillar_cols]
bar_cols2   = [GREEN if s>=20 else (AMBER if s>=10 else RED) for s in scores]
b2 = ax2.bar(pillars, scores, color=bar_cols2, alpha=0.85, width=0.5, zorder=3, edgecolor='white', linewidth=0.5)
for bar, val in zip(b2, scores):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'{int(val)}/{n_total}',
             ha='center', fontsize=12, color='white', fontweight='bold')
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.6, f'({val/n_total*100:.0f}%)',
             ha='center', fontsize=9, color=TEXT_GREY)
ax2.set_ylabel('HBCs with Policy in Place', fontsize=10)
ax2.set_title('UNHLM Commitments:\nHow Many of 30 HBCs Have These Policies?', color=TEXT_WHITE, fontsize=11, pad=8)
ax2.tick_params(length=0); ax2.yaxis.set_visible(False); ax2.set_ylim(0, n_total*1.3)

fig.text(0.5, 0.97, 'UNHLM Accountability: Funding Gaps & Social Protection Commitments',
         ha='center', fontsize=13, fontweight='bold', color=TEXT_WHITE)
fig.text(0.5, 0.93,
         'Where Gates Foundation is substituting for absent domestic investment -- and which governments are viable long-term co-investors',
         ha='center', fontsize=10, color=TEXT_GREY)
fig.text(0.5, 0.01, 'Sources: TB_budget_2024 + TB_unhlm_2024-03-21 (UNHLM file first use in this project)',
         ha='center', fontsize=8, color=TEXT_DIM)
plt.tight_layout(rect=[0, 0.04, 1, 0.92])
plt.savefig(FIG_PATH / '09_unhlm_accountability.png', dpi=180, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print("Saved: figures/09_unhlm_accountability.png")


---
## Section 5 -- The Precision Intervention Matrix
### One Table That Replaces a Hundred Slide Deck

**The synthesis:**
For each of the 30 WHO high-burden countries, the data specifies the intervention
architecture -- not as a generic recommendation, but as a specific prescription derived
from the interaction of biological, structural, and political constraints.

| Intervention | Trigger | Rationale |
|---|---|---|
| **NUTRITION-DX BUNDLE** | und_pct > 35% | Immunological substrate must be addressed first |
| **DUAL HIV+TB MOLECULAR DX** | undercount_gap > 15pp | AI X-ray unreliable; GeneXpert required |
| **PHARMACY AI TRIAGE** | priv_pct > 50% + gender_gap > 3pp | Men seek informal care; diagnostic must follow |
| **DIGITAL TPT TRACKER** | tpt_eff < 30% | 75% of screened contacts never start prevention |
| **EMERGENCY GRANT** | domestic_pct < 30% | Conditionality required for sustainability |

Countries with multiple interventions are not being punished with complexity.
They are where the system is failing on multiple axes simultaneously -- and precisely
where an integrated diagnostic platform has the highest incremental impact.


In [ ]:
# Section 5: Precision Matrix Summary

print("=" * 90)
print("NOTEBOOK 09: PRECISION INTERVENTION MATRIX")
print("=" * 90)

from collections import Counter
all_prescriptions = []
for _, row in eq.iterrows():
    for p in row['prescription'].split(' | '):
        all_prescriptions.append(p.strip())
p_counts = Counter(all_prescriptions)
print("INTERVENTION FREQUENCY (how many of 30 HBCs trigger each):")
for interv, count in sorted(p_counts.items(), key=lambda x: -x[1]):
    bar = chr(9608)*count
    print(f"  {interv:<40} {count:>2}/30  {bar}")
print()
print("=" * 90)
print(f"{'Country':<30} {'Score':>6}  Prescription")
print("-" * 88)
for _, row in eq.iterrows():
    print(f"  {row['country']:<28} {row['equity_score']:>5.1f}  {row['prescription']}")


In [ ]:
# Chart 5: Precision Intervention Matrix

eq_final = eq.sort_values('equity_score', ascending=True).copy()

def primary_colour(row):
    if row.get('und_pct',0) > 35:         return RED
    if row.get('undercount_gap',0) > 15:  return PURPLE
    if row.get('tpt_eff',100) < 30:       return AMBER
    if row.get('domestic_pct',100) < 30:  return '#E05050'
    return TEAL
eq_final['bar_col'] = eq_final.apply(primary_colour, axis=1)

fig, ax = plt.subplots(figsize=(14, 9))
fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(DARK_BG)
y_pos = np.arange(len(eq_final))

ax.barh(y_pos, eq_final['equity_score'], color=eq_final['bar_col'], alpha=0.85, height=0.65, zorder=3)
for i, (_, row) in enumerate(eq_final.iterrows()):
    ax.text(row['equity_score']+0.3, i, f"  {row['equity_score']:.1f}", va='center', fontsize=8, color='white', alpha=0.85)
    rx = row['prescription'][:65] if pd.notna(row['prescription']) else 'STANDARD'
    ax.text(0.5, i, rx, va='center', fontsize=6.5, color='white', alpha=0.65)

ax.set_yticks(y_pos); ax.set_yticklabels(eq_final['country'], fontsize=9.5)
ax.set_xlabel('Equity Vulnerability Score (Notebooks 08-09 composite)', fontsize=10)
ax.tick_params(length=0); ax.yaxis.grid(False); ax.xaxis.grid(True, color='#1E1E2E', lw=0.7)

legend_elems = [
    mpatches.Patch(color=RED,      label='Primary: UNDERNUTRITION -- Nutrition-Dx Bundle', alpha=0.85),
    mpatches.Patch(color=PURPLE,   label='Primary: HIV UNDERCOUNT -- Dual Molecular Dx', alpha=0.85),
    mpatches.Patch(color=AMBER,    label='Primary: TPT FAILURE -- Digital Tracker', alpha=0.85),
    mpatches.Patch(color='#E05050',label='Primary: FUNDING GAP -- Emergency Grant', alpha=0.85),
    mpatches.Patch(color=TEAL,     label='Multi-factorial / Standard targeted', alpha=0.85),
]
ax.legend(handles=legend_elems, fontsize=8, framealpha=0.15, edgecolor='#444', loc='lower right')

fig.text(0.065, 0.97, 'Notebook 09: Precision Intervention Matrix -- 30 High Burden Countries',
         fontsize=14, fontweight='bold', color=TEXT_WHITE)
fig.text(0.065, 0.935,
         'Each country receives a specific intervention architecture. Bar = composite vulnerability score. Label = evidence-based prescription.',
         fontsize=10, color=TEXT_GREY)
fig.text(0.065, 0.01,
         'Synthesis: all 16 WHO datasets used across Notebooks 01-09. No file left behind.',
         fontsize=8, color=TEXT_DIM)
plt.tight_layout(rect=[0, 0.04, 1, 0.93])
plt.savefig(FIG_PATH / '09_precision_intervention_matrix.png', dpi=180, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print("Saved: figures/09_precision_intervention_matrix.png")


In [ ]:
# Final Export

export_cols = ['country','iso3','g_whoregion','equity_score',
               'und_pct','gender_gap','tpt_eff','undercount_gap',
               'priv_pct','domestic_pct','political_will','prescription']
eq[export_cols].to_csv('CARBX_Precision_Intervention_Matrix_2026.csv', index=False)

print("Saved: CARBX_Precision_Intervention_Matrix_2026.csv")
print()
print("=" * 60)
print("COMPLETE PROJECT PIPELINE -- ALL 9 NOTEBOOKS")
print("=" * 60)
steps = [
    ("01", "Data Audit",              "Quality, completeness, 16 WHO datasets"),
    ("02", "Feature Engineering",     "CARB-X Index, master table"),
    ("03", "Market Segmentation",     "K-Means, 4 market archetypes"),
    ("04", "Gates AI Readiness",      "AI Readiness Score, 2x2 matrix"),
    ("05", "2030 Trajectory",         "BAU vs CARB-X forecast"),
    ("06", "Computer Vision",         "ResNet50 Grad-CAM, YOLOv8 AFB"),
    ("07", "Drug Resistance",         "MDR/XDR gap, DST capacity"),
    ("08", "Equity & Prevention",     "Risk factors, gender, TPT, HIV"),
    ("09", "Precision Intervention",  "Reactivation, AI limits, pharmacy, UNHLM  "),
]
for nb, title, desc in steps:
    print(f"  {nb} | {title:<26} {desc}")
print()
print("All 16 WHO datasets used. No file left behind.")
print("Final deliverable: CARBX_Precision_Intervention_Matrix_2026.csv")


---

## Notebook 09 -- What the Complete Pipeline Proves

### Nine notebooks. Sixteen datasets. One strategic question answered.

The question the Gates Foundation asks before anything else: *Does this person
understand where the problem is hard?*

**The reactivation finding** proves understanding of basic TB immunology. In countries
where undernutrition drives >35% of burden, the drug -- whether a diagnostic or a
bactericide -- cannot substitute for T-cell mediated immunity. Nutrition is not a
nice-to-have. It is the clinical prerequisite for every other intervention.

**The AI blind spot finding** is the kind of admission that separates a serious proposal
from a sales document. The model is powerful. It also has limits in specific,
biologically predictable ways. A proposal that doesn't identify those limits cannot
be trusted on anything else.

**The pharmacy channel finding** explains why the male treatment gap exists and what
structural intervention fixes it -- not awareness campaigns, not more clinics, but a
diagnostic that goes to where men already seek care.

**The UNHLM accountability finding** converts the humanitarian argument into a
sustainability argument. DRC at 6.3% domestic funding. Pakistan at 6.9%.
Identifying which governments are co-investors -- and which require conditionality --
is a deployment risk analysis, not a political statement.

---

**The sentence a Gates Foundation programme officer will say after reviewing this pipeline:**

*"She's already done the job."*

---
*Dr. Frida Arrey Takubetang | CARB-X x Gates Foundation Strategic Initiative | 2026*
